# Academic-Data-Agent

## 项目简介

这是一个面向 Hello-Agents 社区展示的精简演示 Notebook，用来快速说明 Academic-Data-Agent 如何处理两类输入：

- 结构化表格数据
- 文本型 PDF 文献

Notebook 默认使用轻量模式，便于社区评审快速复现。

## 作者信息

- GitHub: @healer-666


## 环境配置

首次运行前，请先在项目目录执行：

```bash
pip install -r requirements.txt
```

并根据 `.env.example` 创建 `.env`，填写可用的模型配置。


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
OUTPUT_ROOT.mkdir(exist_ok=True)

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from data_analysis_agent.agent_runner import run_analysis
from data_analysis_agent.presentation import render_trace_table, render_full_report, render_diagnostics


In [2]:
def render_run_summary(result, title: str):
    methods = ", ".join(result.methods_used) if result.methods_used else "unknown"
    tools = ", ".join(result.tools_used) if result.tools_used else "unknown"
    summary_md = f"""
## {title}

- 输入类型: `{result.input_kind}`
- 数据路径: `{result.data_context.absolute_path.as_posix()}`
- 数据规模: `{result.data_context.shape[0]} x {result.data_context.shape[1]}`
- 识别领域: `{result.detected_domain}`
- 使用工具: `{tools}`
- 分析方法: `{methods}`
- 文档解析状态: `{result.document_ingestion_status}`
- 候选表数量: `{result.candidate_table_count}`
- 选中主表: `{result.selected_table_id or 'not_applicable'}`
- PDF 多表综合: `{result.pdf_multi_table_mode}`
- 报告路径: `{result.report_path.as_posix()}`
- Trace 路径: `{result.trace_path.as_posix()}`
- 审稿状态: `{result.review_status}`
- 总耗时: `{result.total_duration_ms / 1000:.2f}s`
"""
    display(Markdown(summary_md))


## 示例 1：表格数据分析

这里使用一个轻量的 Excel 样例，演示标准表格分析路径。社区版默认使用 `draft + auto`，尽量减少等待时间。


In [3]:
tabular_result = run_analysis(
    data_path=PROJECT_ROOT / "data" / "sample_table.xlsx",
    output_dir=OUTPUT_ROOT,
    quality_mode="draft",
    latency_mode="auto",
    verbose=True,
)
render_run_summary(tabular_result, "表格案例运行摘要")


[1/7] Loading runtime configuration...
      Model: deepseek-chat | Tavily credential: detected
      Latency mode: auto
      Vision review: configured
[2/7] Created production run directory...
      Run root: c:/Users/pc/OneDrive/Desktop/agent/Co-creation-projects/healer-666-Academic-Data-Agent/outputs/run_20260317_130434
[3/7] Running input document ingestion...
      Input kind: tabular
      Document ingestion skipped: input is already tabular.
[4/7] Building compact dataset metadata context...
      Data shape: 1070 rows x 31 columns
[5/7] Tool registry ready: PythonInterpreterTool, TavilySearchTool
      Fast path: True | effective max steps: 3
[6/7] Advanced Data Analyst started reasoning (max steps = 3)
      Analysis round: 1
      Step 1/3: thinking...
      Calling PythonInterpreterTool | Stage 1: Load the raw Excel file, inspect data quality, clean missing values and column names, and save cleaned dataset.
      Completed PythonInterpreterTool | status = success
        Ob

      Completed PythonInterpreterTool | status = success
        Observation: Loaded cleaned data shape: (1070, 31) Columns: ['序号', '孕妇代码', '年龄', '身高', '体重', '末次月经', 'IVF妊娠', '检测日期', '检测抽血次数', '检测孕周', '孕妇BMI', '原始读段数', '在参考基因组上比对的比例', '重复读段的比例', '唯一比对的读段数', 'GC含量', '13号染色体的Z值', '18号染色体的Z值', '21号染色
      Step 3/3: thinking...
      Final report generated successfully.
[7/7] Saving Markdown report and run trace...
      Final report: c:/Users/pc/OneDrive/Desktop/agent/Co-creation-projects/healer-666-Academic-Data-Agent/outputs/run_20260317_130434/final_report.md
      Agent trace: c:/Users/pc/OneDrive/Desktop/agent/Co-creation-projects/healer-666-Academic-Data-Agent/outputs/run_20260317_130434/logs/agent_trace.json
[8/8] Production artifact validation passed.



## 表格案例运行摘要

- 输入类型: `tabular`
- 数据路径: `C:/Users/pc/OneDrive/Desktop/agent/Co-creation-projects/healer-666-Academic-Data-Agent/data/sample_table.xlsx`
- 数据规模: `1070 x 31`
- 识别领域: `生物医学/无创产前检测（NIPT）`
- 使用工具: `PythonInterpreterTool`
- 分析方法: `数据清洗（缺失值处理、类型转换）, 描述性统计, Shapiro-Wilk正态性检验, Mann-Whitney U检验, 效应量计算（秩二列相关系数）, Bootstrap置信区间估计, Pearson相关性分析, 箱线图, 热力图, 分布直方图`
- 文档解析状态: `not_needed`
- 候选表数量: `0`
- 选中主表: `not_applicable`
- PDF 多表综合: `False`
- 报告路径: `c:/Users/pc/OneDrive/Desktop/agent/Co-creation-projects/healer-666-Academic-Data-Agent/outputs/run_20260317_130434/final_report.md`
- Trace 路径: `c:/Users/pc/OneDrive/Desktop/agent/Co-creation-projects/healer-666-Academic-Data-Agent/outputs/run_20260317_130434/logs/agent_trace.json`
- 审稿状态: `skipped`
- 总耗时: `110.79s`


In [4]:
render_trace_table(tabular_result)


Step,Stage / Tool,Decision,Status,Short Observation,Notes
1,本地 Python 分析 (PythonInterpreterTool),"Stage 1: Load the raw Excel file, inspect data quality, clean missing values and column names, and save cleaned dataset.",成功,"Loading raw Excel file... Raw shape: (1070, 31) Raw columns: ['序号', '孕妇代码', '年龄', '身高', '体重', '末次月经', 'IVF妊娠', '检测日期', '检测抽血次数', '检测孕周', '孕妇BMI', '原始读段数', '在参考基因组上比对的比例', '重复读段的比例', '唯一比对的读段数 ', 'GC含量', '13号染色体的Z值', '18号","Local Python execution | status=success | decision=Stage 1: Load the raw Excel file, inspect data quality, clean missing values and column names, and save cleaned dataset. | observation=Loading raw Excel file... Raw shape: (1070, 31) Raw columns: ['序号', '孕妇代码', '年龄', '身高', '体重', '末次月经', 'IVF妊娠', '检测日期', '检测抽血次数', '检测孕周', '孕妇BMI', '原始读段数', '在参考基因组上比对的比例', '重复读段的比例', '唯一比对的读段数 ', 'GC含量', '13号染色体的Z值', '18号"
2,本地 Python 分析 (PythonInterpreterTool),"Stage 2: Load cleaned data, perform statistical analysis, generate visualizations, and save figures.",成功,"Loaded cleaned data shape: (1070, 31) Columns: ['序号', '孕妇代码', '年龄', '身高', '体重', '末次月经', 'IVF妊娠', '检测日期', '检测抽血次数', '检测孕周', '孕妇BMI', '原始读段数', '在参考基因组上比对的比例', '重复读段的比例', '唯一比对的读段数', 'GC含量', '13号染色体的Z值', '18号染色体的Z值', '21号染色","Local Python execution | status=success | decision=Stage 2: Load cleaned data, perform statistical analysis, generate visualizations, and save figures. | observation=Loaded cleaned data shape: (1070, 31) Columns: ['序号', '孕妇代码', '年龄', '身高', '体重', '末次月经', 'IVF妊娠', '检测日期', '检测抽血次数', '检测孕周', '孕妇BMI', '原始读段数', '在参考基因组上比对的比例', '重复读段的比例', '唯一比对的读段数', 'GC含量', '13号染色体的Z值', '18号染色体的Z值', '21号染色"
3,最终报告,"Analysis complete. Generate final report with APA-style statistics, effect sizes, and telemetry.",成功,无,"Generated final Markdown report: Analysis complete. Generate final report with APA-style statistics, effect sizes, and telemetry."


## 示例 2：PDF 文献分析

这里使用一篇小型样例论文。当前版本会：

- 抽取文献背景
- 识别候选表
- 自动选择一张主表做定量分析
- 将其他候选表作为上下文证据参与报告解释


In [5]:
pdf_result = run_analysis(
    data_path=PROJECT_ROOT / "data" / "sample_paper.pdf",
    output_dir=OUTPUT_ROOT,
    quality_mode="draft",
    latency_mode="auto",
    document_ingestion_mode="auto",
    verbose=True,
)
render_run_summary(pdf_result, "PDF 案例运行摘要")


[1/7] Loading runtime configuration...
      Model: deepseek-chat | Tavily credential: detected
      Latency mode: auto
      Vision review: configured
[2/7] Created production run directory...
      Run root: c:/Users/pc/OneDrive/Desktop/agent/Co-creation-projects/healer-666-Academic-Data-Agent/outputs/run_20260317_130655
[3/7] Running input document ingestion...
      Input kind: pdf
      Document ingestion completed | status = completed
      Summary: PDF 文档解析完成：共提取 2 张候选表，已选择 table_01 作为主表写入 cleaned_data.csv，其余候选表会与文献背景一起作为综合解释上下文。
[4/7] Building compact dataset metadata context...
      Data shape: 7 rows x 5 columns
[5/7] Tool registry ready: PythonInterpreterTool
      Fast path: True | effective max steps: 3
[6/7] Advanced Data Analyst started reasoning (max steps = 3)
      Analysis round: 1
      Step 1/3: thinking...
      Calling PythonInterpreterTool | Stage 1: Load raw PDF table, inspect, clean column names, handle missing values, save cleaned dataset.
      Completed P


## PDF 案例运行摘要

- 输入类型: `pdf`
- 数据路径: `C:/Users/pc/OneDrive/Desktop/agent/Co-creation-projects/healer-666-Academic-Data-Agent/outputs/run_20260317_130655/data/extracted_tables/table_01.csv`
- 数据规模: `7 x 5`
- 识别领域: `computer_vision_object_detection`
- 使用工具: `PythonInterpreterTool`
- 分析方法: `descriptive_statistics, ranking, bootstrap_confidence_intervals, spearman_correlation, visualization`
- 文档解析状态: `completed`
- 候选表数量: `2`
- 选中主表: `table_01`
- PDF 多表综合: `True`
- 报告路径: `c:/Users/pc/OneDrive/Desktop/agent/Co-creation-projects/healer-666-Academic-Data-Agent/outputs/run_20260317_130655/final_report.md`
- Trace 路径: `c:/Users/pc/OneDrive/Desktop/agent/Co-creation-projects/healer-666-Academic-Data-Agent/outputs/run_20260317_130655/logs/agent_trace.json`
- 审稿状态: `skipped`
- 总耗时: `122.89s`


In [6]:
render_full_report(pdf_result)


## 完整报告正文

# 小样本目标检测模型性能对比分析报告

## 数据概览
本数据集来源于一篇关于目标检测模型改进的学术论文（PDF 提取表）。数据包含 7 种 YOLO 系列模型配置（YOLOv5、YOLOv6、YOLOv7、YOLOv8、YOLOv9、YOLOv11、YOLOv8‑BFDS）在四个关键性能指标上的表现：
- **Precision**（精确率）
- **Recall**（召回率）
- **mAP50**（平均精度，IoU 阈值为 0.5）
- **mAP50‑95**（平均精度，IoU 阈值从 0.5 到 0.95 的平均值）

数据规模为 7 行 × 5 列，属于典型的小样本结果表（N=7）。原始数据无缺失值，已进行列名标准化（将 `mAP50-95` 改为 `mAP50_95`）并保存为清洗后 CSV。

## 方法说明
鉴于样本量极小（N=7），本次分析严格遵循小样本分析原则：
1. **描述性统计**：计算各指标的均值、标准差、四分位数。
2. **排序与排名**：按各指标从高到低排列模型。
3. **Bootstrap 置信区间**：采用非参数 Bootstrap（5000 次重抽样，百分位法）估计各指标总体均值的 95% 置信区间，避免对分布形态的假设。
4. **相关性分析**：使用 Spearman 秩相关系数（非参数）评估指标间的单调关联，同时提供 Pearson 相关系数作为参考。
5. **可视化**：生成 4 幅轻量图表，直观展示模型间性能差异与指标间关系。

**统计学治理说明**：
- 未进行假设检验（如 t 检验、ANOVA），因为每个模型仅有一个观测值，不具备重复测量或实验组内重复，不符合群体比较的前提。
- 所有结论均基于描述性统计与 Bootstrap 区间，避免过度推断。
- 相关性分析仅提示关联，不暗示因果关系。

## 核心假设检验结论
**本分析未执行传统假设检验**，原因如下：
- 每个模型配置仅有一个观测值，无法估计组内变异，因此不能进行组间显著性检验。
- 小样本（N=7）下，任何参数检验的统计功效极低，且正态假设难以验证。
- 遵循《PDF_Small_Table_Mode》指导，仅进行描述性、排序、Bootstrap 区间及相关性分析。

替代的量化结论如下：

### 1. Bootstrap 95% 置信区间（各指标总体均值）
| 指标 | 样本均值 | 95% CI (Bootstrap) |
|------|----------|---------------------|
| Precision | 0.8983 | [0.8462, 0.9501] |
| Recall | 0.8647 | [0.8063, 0.9232] |
| mAP50 | 0.9018 | [0.8519, 0.9516] |
| mAP50_95 | 0.6733 | [0.5996, 0.7471] |

**解读**：所有指标的置信区间均较宽，反映小样本下估计的不确定性。mAP50_95 的区间下限最低（约 0.60），提示该指标在不同模型间波动较大。

### 2. 模型排名（按各指标降序）
- **Precision**：YOLOv8‑BFDS (0.970) > YOLOv5 (0.959) > YOLOv8 (0.950) > YOLOv9 (0.946) > YOLOv11 (0.888) > YOLOv6 (0.840) > YOLOv7 (0.734)
- **Recall**：YOLOv8‑BFDS (0.990) > YOLOv11 (0.899) > YOLOv7 (0.891) > YOLOv5 (0.864) > YOLOv9 (0.808) > YOLOv6 (0.804) > YOLOv8 (0.799)
- **mAP50**：YOLOv8‑BFDS (0.991) > YOLOv11 (0.947) > YOLOv5 (0.938) > YOLOv7 (0.897) > YOLOv8 (0.868) > YOLOv9 (0.855) > YOLOv6 (0.816)
- **mAP50_95**：YOLOv8‑BFDS (0.836) > YOLOv11 (0.699) > YOLOv5 (0.678) > YOLOv7 (0.675) > YOLOv8 (0.622) > YOLOv9 (0.604) > YOLOv6 (0.600)

**综合表现最佳**：YOLOv8‑BFDS 在四项指标上均排名第一，且领先幅度明显。

### 3. Spearman 秩相关系数矩阵
| | Precision | Recall | mAP50 | mAP50_95 |
|-------------|-----------|--------|-------|----------|
| Precision  | 1.000 | 0.179 | 0.536 | 0.714 |
| Recall     | 0.179 | 1.000 | 0.536 | 0.536 |
| mAP50      | 0.536 | 0.536 | 1.000 | 0.893 |
| mAP50_95   | 0.714 | 0.536 | 0.893 | 1.000 |

**解读**：
- mAP50 与 mAP50_95 呈现强正相关（ρ=0.893），说明两者变化趋势高度一致。
- Precision 与 mAP50_95 呈中度正相关（ρ=0.714）。
- Recall 与 Precision 几乎无单调关联（ρ=0.179），提示在这组模型中，高精确率不一定伴随高召回率，反之亦然。

## 结果解释
1. **YOLOv8‑BFDS 全面领先**：该模型在四项指标上均位列第一，尤其 Recall (0.990) 与 mAP50 (0.991) 接近完美，与文献背景中“增强的 YOLOv8 模型集成 DCNv2、E‑SEModule、Concat_BiFPN 等优化模块”的描述相符，说明其改进有效提升了检测精度与召回。
2. **传统模型表现分化**：YOLOv5 在 Precision 和 mAP50 上表现优异（第二、三位），但 Recall 仅居中；YOLOv7 在 Recall 上较高（第三），但 Precision 最低（0.734）。这种分化反映了不同模型架构在精度‑召回权衡上的差异。
3. **mAP50_95 整体较低**：所有模型的 mAP50_95 均低于 0.84，且 Bootstrap 区间下限仅约 0.60，说明在更严格的 IoU 阈值范围内，模型性能仍有较大提升空间。
4. **相关性提示**：mAP50 与 mAP50_95 强相关，可视为一致性验证；Precision 与 Recall 几乎无关，符合目标检测中常存在的“精度‑召回 trade‑off”现象。

## 讨论
### 与文献背景的关联
- 背景文献提到“YOLOv8‑BFDS 集成 DCNv2、E‑SEModule、Concat_BiFPN 等模块以提升对变形、遮挡、小目标及低对比度物体的检测能力”。本数据中 YOLOv8‑BFDS 在 Recall（0.990）和 mAP50（0.991）上显著高于其他模型，间接支持了这些模块的有效性。
- 文献亦提及“CB‑YOLOv5s 通过双向通道解决目标相互遮挡及背景相似导致的低检测精度问题”。本表中未包含 CB‑YOLOv5s，但 YOLOv5 本身在 Precision 和 mAP50 上表现良好，说明基础 YOLOv5 已有较强检测能力，改进版本可能在此基础上进一步提升。

### 方法学局限
1. **样本量极小**：仅 7 个模型，每个模型仅单次观测，无法进行统计检验，所有结论均为描述性。
2. **缺乏重复测量**：未提供同一模型在多组数据上的性能变异，因此无法评估模型稳定性。
3. **数据来源单一**：仅来自一篇论文的单个结果表，可能存在选择性报告偏差。
4. **未控制实验条件**：不同模型的训练数据、超参数、硬件环境可能不同，影响直接可比性。

### 与候选表的交叉验证
PDF 中另一候选表（table_02）包含 YOLOv8 与 YOLOv8+BiFPN 的对比，其中 YOLOv8+BiFPN 的 Precision、Recall、mAP50 分别为 0.973、0.983、0.988，均高于本表中 YOLOv8 的对应值（0.950、0.799、0.868）。这进一步支持了 BiFPN 结构对性能的提升作用，与本表中 YOLOv8‑BFDS（含 Concat_BiFPN）表现最优的趋势一致。

## 清洗后数据路径
`c:/Users/pc/OneDrive/Desktop/agent/Co-creation-projects/healer-666-Academic-Data-Agent/outputs/run_20260317_130655/data/cleaned_data.csv`

## 图表引用
1. **各模型精确率与召回率对比**  
![精确率与召回率对比](c:/Users/pc/OneDrive/Desktop/agent/Co-creation-projects/healer-666-Academic-Data-Agent/outputs/run_20260317_130655/figures/review_round_1/bar_precision_recall.png)
2. **精确率 vs 召回率散点图**  
![精确率 vs 召回率散点图](c:/Users/pc/OneDrive/Desktop/agent/Co-creation-projects/healer-666-Academic-Data-Agent/outputs/run_20260317_130655/figures/review_round_1/scatter_precision_recall.png)
3. **Spearman 秩相关热图**  
![Spearman 秩相关热图](c:/Users/pc/OneDrive/Desktop/agent/Co-creation-projects/healer-666-Academic-Data-Agent/outputs/run_20260317_130655/figures/review_round_1/heatmap_spearman_corr.png)
4. **多指标平行坐标图**  
![多指标平行坐标图](c:/Users/pc/OneDrive/Desktop/agent/Co-creation-projects/healer-666-Academic-Data-Agent/outputs/run_20260317_130655/figures/review_round_1/parallel_coordinates.png)

In [7]:
render_diagnostics(pdf_result)


## 总结与展望

这个社区版演示重点展示三件事：

- 结构化表格的自动分析能力
- PDF 文献中的候选表提取与主表分析能力
- 运行产物、报告与 trace 的可追踪性

完整版项目还提供了 Gradio 工作台、历史记录浏览、视觉审稿和更完整的工程能力。
